# 01 — PPO Training (3D Pack)

Trains PPO on the 3D cell-resolved battery pack thermal environment.

- Algorithm: **Proximal Policy Optimization** (on-policy, VecNormalize)
- Environment: 4×3×2 cell pack, 7-element observation, continuous cooling action
- Full training: 3 000 000 timesteps (~1–2 h on T4 GPU)

**Run `00_colab_setup.ipynb` first.**

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Adjust path if needed
%cd /content/drive/MyDrive/RL-Battery-Thermal-Management

import os, sys
PROJECT_ROOT = os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

!pip install -q -r requirements.txt

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## Save Paths

In [ ]:
BASE_DRIVE_DIR = '/content/drive/MyDrive/battery_rl'

PPO_SAVE_DIR = f'{BASE_DRIVE_DIR}/models/ppo_3d_pack'
PPO_LOG_DIR  = f'{BASE_DRIVE_DIR}/logs/ppo_3d_pack'

os.makedirs(PPO_SAVE_DIR, exist_ok=True)
os.makedirs(PPO_LOG_DIR,  exist_ok=True)

print('Models will save to:', PPO_SAVE_DIR)
print('Logs will save to:  ', PPO_LOG_DIR)

## Smoke Test (10 000 steps)

Run this first to confirm the pipeline works before launching full training.

In [ ]:
!python training/train_pack_ppo_3d.py \
  --timesteps 10000 \
  --n-envs 2 \
  --save-dir "$PPO_SAVE_DIR" \
  --log-dir  "$PPO_LOG_DIR"

In [ ]:
print('Smoke test outputs:')
for f in sorted(os.listdir(PPO_SAVE_DIR)):
    print(' ', f)

## Full Training (3 000 000 steps)

Checkpoints save every 50 000 steps. If Colab disconnects, resume with
`--resume-from $PPO_SAVE_DIR/checkpoints/ppo_pack_checkpoint_<N>_steps.zip`

In [ ]:
!python training/train_pack_ppo_3d.py \
  --timesteps 3000000 \
  --n-envs 8 \
  --save-dir "$PPO_SAVE_DIR" \
  --log-dir  "$PPO_LOG_DIR" \
  --device auto

## Resume from Checkpoint (if interrupted)

In [ ]:
# List available checkpoints
ckpt_dir = os.path.join(PPO_SAVE_DIR, 'checkpoints')
if os.path.exists(ckpt_dir):
    checkpoints = sorted(os.listdir(ckpt_dir))
    print('Available checkpoints:')
    for c in checkpoints:
        print(' ', c)
    if checkpoints:
        latest = os.path.join(ckpt_dir, checkpoints[-1])
        print('\nLatest checkpoint:', latest)
else:
    print('No checkpoints found.')

In [ ]:
# Uncomment and set RESUME_FROM to your checkpoint path:
# RESUME_FROM = f'{PPO_SAVE_DIR}/checkpoints/ppo_pack_checkpoint_XXXXXX_steps.zip'
#
# !python training/train_pack_ppo_3d.py \
#   --timesteps 3000000 \
#   --n-envs 8 \
#   --save-dir "$PPO_SAVE_DIR" \
#   --log-dir  "$PPO_LOG_DIR" \
#   --resume-from "$RESUME_FROM"

## TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir "$PPO_LOG_DIR"

## Verify Saved Models

In [ ]:
print('Models saved:')
for root, dirs, files in os.walk(PPO_SAVE_DIR):
    for f in sorted(files):
        rel = os.path.relpath(os.path.join(root, f), PPO_SAVE_DIR)
        print(' ', rel)